# 4.2 Cálculo de Capacitancia

Para calcular la capacitancia de un arreglo geométrico, generalmente asumimos una carga $Q$, calculamos el campo eléctrico $\vec{E}$ mediante la ley de Gauss, determinamos la diferencia de potencial $\Delta V = -\int \vec{E} \cdot d\vec{l}$ y finalmente aplicamos la definición $C = Q / \Delta V$.

## Condensador esférico

Consideremos dos cascarones esféricos conductores concéntricos de radios $a$ (interno) y $b$ (externo).


&nbsp;
```{image} images_tikz/04_02.svg
:alt: Condensador esférico
:width: 40%
:align: center
```
&nbsp;

In [11]:
import numpy as np
import plotly.graph_objects as go

# Función geométrica para crear una esfera (superficie)
def crear_esfera(radio, resolucion=50):
    # Generar la rejilla esférica (phi: [0, pi], theta: [0, 2*pi])
    phi = np.linspace(0, np.pi, resolucion)
    theta = np.linspace(0, 2 * np.pi, resolucion)
    phi_grid, theta_grid = np.meshgrid(phi, theta)
    
    # Coordenadas cartesianas
    x = radio * np.sin(phi_grid) * np.cos(theta_grid)
    y = radio * np.sin(phi_grid) * np.sin(theta_grid)
    z = radio * np.cos(phi_grid)
    return x, y, z

# --- PARÁMETROS INICIALES ---
# Define los radios (asegúrate de que rext > rint)
r_interior = 2.0  # Esfera interior sólida
r_exterior = 3.5  # Cascarón exterior translúcido

# Generar coordenadas para ambas esferas
x_int, y_int, z_int = crear_esfera(r_interior)
x_ext, y_ext, z_ext = crear_esfera(r_exterior)

# Crear la figura interactiva 3D
fig = go.Figure()

# --- AÑADIR SUPERFICIES ---

# 1. Esfera Interior (Sólida - ej. Cobre/Naranja)
fig.add_trace(go.Surface(
    x=x_int, y=y_int, z=z_int,
    colorscale='Oranges', # Estilo visual 'cobre'
    showscale=False,
    opacity=1.0,         # Completamente opaca
    name='Esfera Interior Sólida'
))

# 2. Cascarón Exterior (Translúcido - ej. Malla/Gris)
fig.add_trace(go.Surface(
    x=x_ext, y=y_ext, z=z_ext,
    colorscale='Greys',   # Estilo visual 'malla metálica'
    showscale=False,
    opacity=0.3,         # Baja opacidad para ver el interior
    name='Cascarón Translúcido'
))

# --- AJUSTES DE DISEÑO Y PROPORCIÓN ---
fig.update_layout(
    title=f'Visualización de Cascarón Esférico (r₁={r_interior}, r₂={r_exterior})',
    scene=dict(
        xaxis_title='Eje X',
        yaxis_title='Eje Y',
        zaxis_title='Eje Z',
        aspectmode='data' # CRÍTICO: Mantiene la proporción geométrica real 1:1:1
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Mostrar la visualización interactiva
fig.show()


La diferencia de potencial es:

$$\Delta V = -\int_{b}^{a} \frac{Q}{4\pi\epsilon_0 r^2} \hat{r} \cdot dr\hat{r} = \frac{Q}{4\pi\epsilon_0} \left( \frac{1}{a} - \frac{1}{b} \right) = \frac{Q}{4\pi\epsilon_0} \frac{(b-a)}{ab}$$

Aplicando la definición de capacitancia, obtenemos:

$$C_{\text{esférico}} = \frac{Q}{\Delta V} = 4\pi\epsilon_0 \frac{ab}{(b-a)}$$

## Condensador cilíndrico

Para un arreglo de dos cilindros coaxiales de longitud $L$ y radios $a$ y $b$ ($L \gg b$), el campo eléctrico en la región intermedia es $\vec{E} = \frac{Q/L}{2\pi\epsilon_0 r}\hat{r}$.


&nbsp;
```{image} images_tikz/04_03.svg
:alt: Condensador cilíndrico
:width: 40%
:align: center
```
&nbsp;

&nbsp;
```{image} images_tikz/04_02.svg
:alt: Vista superior
:width: 40%
:align: center
```
&nbsp;

In [12]:
from IPython.display import display, HTML

simulacion_condensador_html = """
<div style="border: 1px solid #e0e0e0; padding: 20px; border-radius: 8px; background-color: #f8f9fa; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial, sans-serif;">
    <h3 style="margin-top:0; color: #2c3e50;">Simulación 2D: Condensador Cilíndrico (Vista Transversal)</h3>
    <p style="font-size: 0.95em; color: #555; margin-bottom: 15px;">Modifica los radios para observar cómo cambia la densidad de las líneas de campo eléctrico y la capacitancia del sistema.</p>
    
    <div style="display: flex; gap: 20px; flex-wrap: wrap; margin-bottom: 15px;">
        <div style="flex: 1; min-width: 250px;">
            <div style="margin-bottom: 8px;">
                <label style="font-weight: 600; display: inline-block; width: 160px; color: #dc3545;">Radio interior a (mm): <span id="cond_val_a" style="font-weight: normal;">5</span></label>
                <input type="range" id="cond_slider_a" min="1" max="15" step="0.5" value="5" style="width: 45%; vertical-align: middle;">
            </div>
            <div>
                <label style="font-weight: 600; display: inline-block; width: 160px; color: #0d6efd;">Radio exterior b (mm): <span id="cond_val_b" style="font-weight: normal;">15</span></label>
                <input type="range" id="cond_slider_b" min="5" max="30" step="0.5" value="15" style="width: 45%; vertical-align: middle;">
            </div>
        </div>

        <div style="flex: 1; min-width: 250px; font-size: 0.95em; background: #fff; padding: 10px; border-radius: 6px; border: 1px solid #ccc;">
            <div style="margin-bottom: 5px;"><b>Capacitancia (por metro):</b> <span id="cond_val_C" style="color: #198754; font-weight: bold;">0</span> pF/m</div>
            <div><b>Campo E máx (en r=a):</b> <span id="cond_val_E" style="color: #dc3545; font-weight: bold;">0</span> V/m</div>
            <div style="font-size: 0.8em; color: #777; margin-top: 5px;">*Asumiendo una diferencia de potencial de 10V</div>
        </div>
    </div>
    
    <div style="text-align: center; position: relative;">
        <canvas id="cond_canvas" width="600" height="400" style="background: white; border: 1px solid #ccc; border-radius: 4px; box-shadow: 0 2px 5px rgba(0,0,0,0.05); max-width: 100%;"></canvas>
    </div>
</div>

<script>
    (function() {
        const canvas = document.getElementById('cond_canvas');
        const ctx = canvas.getContext('2d');
        
        const slider_a = document.getElementById('cond_slider_a');
        const slider_b = document.getElementById('cond_slider_b');
        const val_a = document.getElementById('cond_val_a');
        const val_b = document.getElementById('cond_val_b');
        const val_C = document.getElementById('cond_val_C');
        const val_E = document.getElementById('cond_val_E');
        
        const epsilon_0 = 8.854e-12;
        const V = 10; // Voltaje fijo para la simulación

        function drawArrow(fromx, fromy, tox, toy, color, width) {
            ctx.beginPath();
            ctx.moveTo(fromx, fromy);
            ctx.lineTo(tox, toy);
            ctx.strokeStyle = color;
            ctx.lineWidth = width;
            ctx.stroke();

            let angle = Math.atan2(toy - fromy, tox - fromx);
            let dist = Math.hypot(tox - fromx, toy - fromy);
            if (dist < 5) return; 
            
            let headlen = 8;
            ctx.beginPath();
            ctx.moveTo(tox, toy);
            ctx.lineTo(tox - headlen * Math.cos(angle - Math.PI/7), toy - headlen * Math.sin(angle - Math.PI/7));
            ctx.lineTo(tox - headlen * Math.cos(angle + Math.PI/7), toy - headlen * Math.sin(angle + Math.PI/7));
            ctx.lineTo(tox, toy);
            ctx.fillStyle = color;
            ctx.fill();
        }

        function updateSim() {
            let a = parseFloat(slider_a.value);
            let b = parseFloat(slider_b.value);
            
            // Lógica para asegurar que b > a
            if (a >= b) {
                b = a + 1;
                slider_b.value = b;
            }
            
            val_a.innerText = a.toFixed(1);
            val_b.innerText = b.toFixed(1);

            // Cálculos físicos
            let a_m = a / 1000;
            let b_m = b / 1000;
            
            // Capacitancia por unidad de longitud C/L = 2*pi*e0 / ln(b/a)
            let C_L = (2 * Math.PI * epsilon_0) / Math.log(b_m / a_m);
            val_C.innerText = (C_L * 1e12).toFixed(2); // Convertir a pF
            
            // Campo eléctrico máximo E = V / (a * ln(b/a))
            let E_max = V / (a_m * Math.log(b_m / a_m));
            val_E.innerText = E_max.toFixed(0);

            // Dibujo
            ctx.clearRect(0, 0, canvas.width, canvas.height);
            let cx = canvas.width / 2;
            let cy = canvas.height / 2;
            
            // Escala visual (30mm max radius = 180px)
            let scale = 180 / 30; 
            let r_a_px = a * scale;
            let r_b_px = b * scale;

            // Dibujar cilindro exterior (Cátodo)
            ctx.beginPath();
            ctx.arc(cx, cy, r_b_px, 0, 2 * Math.PI);
            ctx.fillStyle = 'rgba(13, 110, 253, 0.1)'; // Azul claro
            ctx.fill();
            ctx.lineWidth = 4;
            ctx.strokeStyle = '#0d6efd';
            ctx.stroke();

            // Dibujar líneas de campo (Vectores)
            let num_lines = 16;
            for(let i = 0; i < num_lines; i++) {
                let angle = (i * 2 * Math.PI) / num_lines;
                
                // Dividimos el espacio en pequeños segmentos para mostrar que la flecha es más grande cerca del centro
                let r_current = a;
                let step = (b - a) / 4;
                
                for(let j=0; j<4; j++) {
                    let r1 = r_current;
                    let r2 = r_current + step * 0.7; // El 0.7 deja un pequeño espacio entre flechas
                    
                    let E_mag = 1 / r1; // Relación proporcional inversa
                    let arrow_width = Math.max(1, E_mag * a * 2); // Grosor depende de la magnitud
                    let alpha = Math.max(0.2, E_mag * a); // Opacidad depende de la magnitud
                    
                    let x1 = cx + r1 * scale * Math.cos(angle);
                    let y1 = cy + r1 * scale * Math.sin(angle);
                    let x2 = cx + r2 * scale * Math.cos(angle);
                    let y2 = cy + r2 * scale * Math.sin(angle);
                    
                    drawArrow(x1, y1, x2, y2, `rgba(44, 62, 80, ${alpha})`, arrow_width);
                    r_current += step;
                }
            }

            // Dibujar cilindro interior (Ánodo)
            ctx.beginPath();
            ctx.arc(cx, cy, r_a_px, 0, 2 * Math.PI);
            ctx.fillStyle = '#dc3545'; // Rojo sólido
            ctx.fill();
            ctx.lineWidth = 2;
            ctx.strokeStyle = '#842029';
            ctx.stroke();
            
            // Etiquetas de cargas
            ctx.fillStyle = 'white';
            ctx.font = 'bold 14px Arial';
            ctx.fillText("+", cx - 4, cy + 5);
            ctx.fillStyle = '#0d6efd';
            ctx.fillText("-", cx + r_b_px + 5, cy + 5);
        }

        slider_a.addEventListener('input', updateSim);
        slider_b.addEventListener('input', updateSim);

        updateSim();
    })();
</script>
"""

display(HTML(simulacion_condensador_html))

In [2]:
import numpy as np
import plotly.graph_objects as go

# Función geométrica para crear los cilindros
def crear_cilindro(radio, longitud, resolucion=50):
    theta = np.linspace(0, 2 * np.pi, resolucion)
    z = np.linspace(0, longitud, 2)
    theta_grid, z_grid = np.meshgrid(theta, z)
    x = radio * np.cos(theta_grid)
    y = radio * np.sin(theta_grid)
    return x, y, z_grid

# Parámetros del cable
longitud_L = 10
radio_interior = 1.0
radio_exterior = 3.0

# Generar coordenadas
x_int, y_int, z_int = crear_cilindro(radio_interior, longitud_L)
x_ext, y_ext, z_ext = crear_cilindro(radio_exterior, longitud_L)

# Crear la figura interactiva
fig = go.Figure()

# Añadir el conductor central (Cobre, opaco)
fig.add_trace(go.Surface(
    x=x_int, y=y_int, z=z_int,
    colorscale='Oranges',
    showscale=False,
    opacity=1.0,
    name='Conductor Central'
))

# Añadir la malla exterior (Gris, translúcida)
fig.add_trace(go.Surface(
    x=x_ext, y=y_ext, z=z_ext,
    colorscale='Greys',
    showscale=False,
    opacity=0.3, # Baja opacidad para ver el interior
    name='Malla Exterior'
))

# Ajustar la vista y proporciones
fig.update_layout(
    title='Sección de Cable Coaxial',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z (Longitud)',
        aspectmode='data' # Mantiene la proporción real de los radios
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

# Mostrar el resultado
fig.show(renderer="iframe")


La diferencia de potencial resulta:

$$\Delta V = -\int_{b}^{a} \frac{Q/L}{2\pi\epsilon_0 r} dr = \frac{Q/L}{2\pi\epsilon_0} \ln\left(\frac{b}{a}\right)$$

Por lo tanto, la capacitancia es:

$$C_{\text{cilíndrico}} = \frac{2\pi\epsilon_0 L}{\ln(b/a)}$$

## Condensador de placas paralelas

Consiste en dos placas metálicas paralelas de área $A$ separadas por una distancia $d$. El campo eléctrico, considerado uniforme entre las placas, tiene una magnitud de $E = \frac{\sigma}{\epsilon_0}$, donde $\sigma = Q/A$.


&nbsp;
```{image} images_tikz/04_04.svg
:alt: Condensador de placas paralelas
:width: 40%
:align: center
```
&nbsp;

&nbsp;
```{image} images_tikz/04_05.svg
:alt: Vista frontal
:width: 30%
:align: center
```
&nbsp;


La diferencia de potencial es simplemente $\Delta V = Ed = \frac{Q}{\epsilon_0 A}d$. La capacitancia es entonces:

$$C_{\text{placas}} = \frac{\epsilon_0 A}{d}$$
